# Clean Up Noisy Speech

Add noise to a clean recording at different SNR levels, run MMSE speech enhancement, and measure the improvement.

In [ ]:
# Install dependencies if needed (e.g. on Colab)
# %pip install pyvoicebox

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from pyvoicebox import v_addnoise, v_ssubmmse, v_snrseg, v_enframe, v_rfft, v_windows

# Download
url = "http://festvox.org/cmu_arctic/cmu_arctic/cmu_us_bdl_arctic/wav/arctic_a0005.wav"
wav_path = "arctic_a0005.wav"
if not os.path.exists(wav_path):
    print("Downloading...")
    urllib.request.urlretrieve(url, wav_path)

clean, fs = sf.read(wav_path)
print(f"Loaded: {fs} Hz, {len(clean)/fs:.2f}s")
print("Content: \"For the twentieth time that evening the two men shook hands.\"")
display(Audio(clean, rate=fs))

## Helper: build a spectrogram

In [ ]:
def spectrogram(sig, fs, frame_ms=25, hop_ms=10):
    """Compute log-magnitude spectrogram using pyvoicebox."""
    frame_len = int(frame_ms * fs / 1000)
    hop_len = int(hop_ms * fs / 1000)
    frames, _, _ = v_enframe(sig, frame_len, hop_len)
    win = v_windows(3, frame_len).flatten()
    spec = np.array([v_rfft(f * win) for f in frames])
    db = 20 * np.log10(np.abs(spec) + 1e-10)
    t = np.arange(db.shape[0]) * hop_len / fs
    f = np.linspace(0, fs / 2, db.shape[1])
    return db, t, f

## Add noise at different SNR levels

`v_addnoise` adds white noise to achieve a target SNR.

In [ ]:
noisy_0db, _ = v_addnoise(clean, fs, 0, 'k')
noisy_5db, _ = v_addnoise(clean, fs, 5, 'k')
noisy_10db, _ = v_addnoise(clean, fs, 10, 'k')

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, (sig, title) in zip(axes.flat, [
    (clean, 'Clean'),
    (noisy_0db, 'Noisy (0 dB SNR)'),
    (noisy_5db, 'Noisy (5 dB SNR)'),
    (noisy_10db, 'Noisy (10 dB SNR)'),
]):
    db, t, f = spectrogram(sig, fs)
    ax.pcolormesh(t, f, db.T, cmap='magma', shading='auto', vmin=-60, vmax=20)
    ax.set_title(title)
    ax.set_ylim(0, fs / 2)
    ax.set_ylabel('Hz')

axes[1, 0].set_xlabel('Time (s)')
axes[1, 1].set_xlabel('Time (s)')
plt.suptitle('Effect of noise at different SNR levels', fontsize=14)
plt.tight_layout()
plt.show()

## MMSE speech enhancement

`v_ssubmmse` implements the MMSE spectral amplitude estimator. It estimates the clean speech spectrum from the noisy input by modelling speech and noise statistics per frequency bin.

Compare the three spectrograms below: the enhanced version should recover the formant structure (horizontal bands) while suppressing the noise floor (the diffuse background energy).

In [ ]:
enhanced = v_ssubmmse(noisy_5db, fs)
print(f"Enhanced signal: {len(enhanced)} samples")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (sig, title, color) in zip(axes, [
    (clean, 'Clean (reference)', '#4caf50'),
    (noisy_5db, 'Noisy (5 dB SNR)', '#f44336'),
    (enhanced, 'Enhanced (MMSE)', '#2196f3'),
]):
    db, t, f = spectrogram(sig, fs)
    ax.pcolormesh(t, f, db.T, cmap='magma', shading='auto', vmin=-60, vmax=20)
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylim(0, fs / 2)

axes[0].set_ylabel('Frequency (Hz)')
plt.suptitle('Speech enhancement: before and after', fontsize=14)
plt.tight_layout()
plt.show()

## Listen and compare

Hear the difference — clean, noisy, and enhanced:

In [ ]:
print("Clean:")
display(Audio(clean, rate=fs))
print("\nNoisy (5 dB SNR):")
display(Audio(noisy_5db, rate=fs))
print("\nEnhanced (MMSE):")
display(Audio(enhanced, rate=fs))

## Measure improvement

`v_snrseg` computes segmental and global SNR by comparing a signal against a reference.

**Segmental SNR** averages SNR across short frames — it better reflects perceptual quality since a single loud noise burst doesn't dominate the average. **Global SNR** is the overall ratio. Both should increase after enhancement.

In [ ]:
# Trim to same length for comparison
min_len = min(len(clean), len(noisy_5db), len(enhanced))
c, n, e = clean[:min_len], noisy_5db[:min_len], enhanced[:min_len]

seg_before, glob_before, _, _, _ = v_snrseg(n, c, fs)
seg_after, glob_after, _, _, _ = v_snrseg(e, c, fs)

print(f"{'':20s} {'Segmental SNR':>15s} {'Global SNR':>12s}")
print(f"{'Noisy (5 dB)':20s} {float(seg_before):>14.1f} dB {float(glob_before):>11.1f} dB")
print(f"{'Enhanced (MMSE)':20s} {float(seg_after):>14.1f} dB {float(glob_after):>11.1f} dB")
print(f"{'Improvement':20s} {float(seg_after - seg_before):>+14.1f} dB {float(glob_after - glob_before):>+11.1f} dB")